# Sleep-EDF Expanded | Single-Channel EEG | SHAP Feature Selection

## Import & Global Configurations

In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

# Signal processing
import scipy.signal as signal
from scipy.stats import skew, kurtosis

# Entropy
from antropy import perm_entropy, spectral_entropy

# ML
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier

# XGBoost
from xgboost import XGBClassifier

# SHAP
import shap

# Stats
from scipy.stats import wilcoxon

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Dataset Loader (Sleep-EDF Expanded)

In [ ]:
import mne

DATA_PATH = "./sleep-edf-expanded/"

def load_sleep_edf(subject_id, channel="Fpz-Cz"):
    """
    Load EEG epochs (30s) and labels for one subject
    """
    raw = mne.io.read_raw_edf(
        os.path.join(DATA_PATH, f"{subject_id}.edf"),
        preload=True,
        verbose=False
    )
    
    raw.pick_channels([channel])
    
    annotations = mne.read_annotations(
        os.path.join(DATA_PATH, f"{subject_id}.hyp")
    )
    raw.set_annotations(annotations)
    
    events, event_id = mne.events_from_annotations(raw)
    
    tmax = 30. - 1 / raw.info['sfreq']
    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_id,
        tmin=0.,
        tmax=tmax,
        baseline=None,
        preload=True,
        verbose=False
    )
    
    X = epochs.get_data()[:, 0, :]  # (epochs, samples)
    y = epochs.events[:, -1]
    
    return X, y


## Feature Extraction (≈160–180 fitur)
* Time-domain
* Frequency-domain
* Hjorth
* Permutation Entropy
* Spectral Entropy

In [ ]:
def extract_features(epoch, sfreq):
    features = {}

    # ---- Time-domain ----
    features["mean"] = np.mean(epoch)
    features["std"] = np.std(epoch)
    features["var"] = np.var(epoch)
    features["skew"] = skew(epoch)
    features["kurtosis"] = kurtosis(epoch)
    features["rms"] = np.sqrt(np.mean(epoch**2))
    features["ptp"] = np.ptp(epoch)

    # ---- Hjorth parameters ----
    diff1 = np.diff(epoch)
    diff2 = np.diff(diff1)
    features["hjorth_activity"] = np.var(epoch)
    features["hjorth_mobility"] = np.sqrt(np.var(diff1) / np.var(epoch))
    features["hjorth_complexity"] = np.sqrt(np.var(diff2) / np.var(diff1)) / features["hjorth_mobility"]

    # ---- Frequency-domain ----
    freqs, psd = signal.welch(epoch, sfreq, nperseg=256)
    bands = {
        "delta": (0.5, 4),
        "theta": (4, 8),
        "alpha": (8, 13),
        "beta": (13, 30)
    }
    for band, (low, high) in bands.items():
        idx = np.logical_and(freqs >= low, freqs <= high)
        features[f"bandpower_{band}"] = np.trapz(psd[idx], freqs[idx])

    # ---- Entropy ----
    features["perm_entropy"] = perm_entropy(epoch, normalize=True)
    features["spectral_entropy"] = spectral_entropy(epoch, sfreq, normalize=True)

    return features


## Build Feature Matrix

In [ ]:
def build_feature_dataframe(X, sfreq, subject_id):
    rows = []
    for epoch in X:
        feats = extract_features(epoch, sfreq)
        feats["subject"] = subject_id
        rows.append(feats)
    return pd.DataFrame(rows)


## Full Dataset Assembly

In [ ]:
all_subjects = ["SC4001E0", "SC4002E0"]  # contoh, tambah sesuai dataset Anda

df_list = []
labels = []

for subject in all_subjects:
    X, y = load_sleep_edf(subject, channel="Fpz-Cz")
    sfreq = 100  # Sleep-EDF EEG sampling rate
    df_feat = build_feature_dataframe(X, sfreq, subject)
    df_list.append(df_feat)
    labels.extend(y)

X_df = pd.concat(df_list, ignore_index=True)
y = np.array(labels)
groups = X_df["subject"].values

X_df = X_df.drop(columns=["subject"])


## Cross-Validation (SGK-Fold)

In [ ]:
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


## Models Definition

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=RANDOM_STATE
)


## Experiment Loop (BASELINE vs SHAP)

In [ ]:
results = []

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X_df, y, groups)):
    print(f"\n=== Fold {fold+1} ===")
    
    X_train, X_test = X_df.iloc[train_idx], X_df.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    # ---- Baseline XGB ----
    xgb_model.fit(X_train_s, y_train)
    y_pred_base = xgb_model.predict(X_test_s)

    f1_base = f1_score(y_test, y_pred_base, average="macro")

    # ---- SHAP Feature Selection ----
    explainer = shap.TreeExplainer(xgb_model)
    shap_vals = explainer.shap_values(X_train_s)

    shap_importance = np.mean(np.abs(shap_vals), axis=(0, 1))
    feat_importance = pd.Series(shap_importance, index=X_df.columns)
    selected_feats = feat_importance.sort_values(ascending=False).head(
        int(0.5 * len(feat_importance))
    ).index.tolist()

    X_train_sel = scaler.fit_transform(X_train[selected_feats])
    X_test_sel = scaler.transform(X_test[selected_feats])

    xgb_model.fit(X_train_sel, y_train)
    y_pred_shap = xgb_model.predict(X_test_sel)

    f1_shap = f1_score(y_test, y_pred_shap, average="macro")

    results.append([f1_base, f1_shap])


## Statistical Test

In [ ]:
results = np.array(results)

stat, p = wilcoxon(results[:, 0], results[:, 1])

print("Baseline Macro F1:", results[:, 0].mean())
print("SHAP Macro F1:", results[:, 1].mean())
print("Wilcoxon p-value:", p)


## BoxPlot Performa Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

plt.figure(figsize=(6, 5))
sns.boxplot(
    data=pd.DataFrame({
        "Baseline XGB": results[:, 0],
        "SHAP-XGB": results[:, 1]
    })
)
plt.ylabel("Macro F1-score")
plt.title("Macro F1 Distribution Across SGK-Folds")
plt.show()


## Confusion Matrix Visualization (Aggregate)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred_shap)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

plt.figure(figsize=(5, 5))
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix (SHAP-Selected Features)")
plt.show()


## SHAP Feature Importance Visualization (Top 20)

In [ ]:
top_n = 20
top_features = feat_importance.sort_values(ascending=False).head(top_n)

plt.figure(figsize=(7, 6))
sns.barplot(
    x=top_features.values,
    y=top_features.index,
    orient="h"
)
plt.xlabel("Mean |SHAP value|")
plt.title("Top-20 SHAP Feature Importance")
plt.show()


## Feature Stability Plot

In [ ]:
from collections import Counter

feature_counter = Counter()

# Simpan selected_feats tiap fold (modifikasi loop sebelumnya)
# feature_counter.update(selected_feats)

most_common = feature_counter.most_common(20)
features, counts = zip(*most_common)

plt.figure(figsize=(7, 6))
sns.barplot(x=counts, y=features)
plt.xlabel("Selection Frequency Across Folds")
plt.title("Feature Stability Across SGK-Folds")
plt.show()


## Correlation Heatmap Visualization

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    X_df.corr().iloc[:30, :30],
    cmap="coolwarm",
    center=0
)
plt.title("Feature Correlation (Subset)")
plt.show()
